# Практика 9. Преобразование, объединение и агрегирование данных

На практике 8 мы привели данные в порядок. Теперь из них нужно получить ответы: кто из
перевозчиков работает лучше, какие маршруты приносят выручку, выполнен ли план.

Для этого данных одной таблицы не хватает. Рейсы лежат отдельно за апрель и май, рейтинги
перевозчиков — в справочнике, сегменты клиентов — в другом, план — в третьем. Сегодня
учимся собирать это вместе и сворачивать в сводные таблицы.

| Часть | О чём |
|---|---|
| 1 | `concat` — склейка таблиц одинаковой структуры |
| 2 | Сортировка, фильтрация, преобразование таблиц |
| 3 | `groupby` и агрегирующие функции |
| 4 | `merge` — соединение по ключу |
| 5 | `pivot_table` и сводные представления |
| 6 | Мини-проект: план/факт по маршрутам |

> **Главная мысль пары:** соединение таблиц — самая опасная операция в аналитике.
> Она умеет **молча менять число строк**, и тогда все суммы становятся неверными.
> В части 4 вы увидите, как одна лишняя строка в справочнике завышает выручку на 3.7 млн
> рублей. Поэтому правило: после каждого соединения — проверка числа строк.

---
## Шаг 0. Проект

```
# терминал
mkdir pandas-aggregation
cd pandas-aggregation
```

`requirements.txt`:

```
pandas
```

```
# терминал (Windows)
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r requirements.txt
```

```
# терминал (macOS / Linux)
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

---
## Данные

Запустите ячейку — она создаст пять таблиц. Это типичная ситуация: данные о событиях
отдельно, справочники отдельно, план отдельно.

| Файл | Что внутри | Зачем |
|---|---|---|
| `trips_april.csv` | рейсы за апрель | склеиваем с майскими |
| `trips_may.csv` | рейсы за май | склеиваем с апрельскими |
| `carriers.csv` | перевозчики: город, парк машин, рейтинг | присоединяем к рейсам |
| `clients.csv` | клиенты: сегмент, менеджер, год начала работы | присоединяем к рейсам |
| `plan.csv` | план по маршрутам на май | сравниваем с фактом |

In [ ]:
# ЯЧЕЙКА-ГЕНЕРАТОР ДАННЫХ. Запустите её (Shift+Enter), менять ничего не нужно.
# Создаёт пять таблиц: рейсы за апрель и май, справочники перевозчиков и клиентов, план по маршрутам.

import csv
import random
from pathlib import Path

random.seed(9)
Path("data").mkdir(exist_ok=True)

routes = {
    "Москва-Казань": 820, "Москва-Санкт-Петербург": 710, "Москва-Нижний Новгород": 420,
    "Москва-Воронеж": 520, "Казань-Самара": 360, "Екатеринбург-Челябинск": 220,
    "Новосибирск-Омск": 680, "Санкт-Петербург-Петрозаводск": 430,
}
carriers = ["ТрансЛогистик", "РусКарго", "СибПуть", "ВолгаТранс", "АвтоДорСервис"]
tariffs = {"Контейнер": 7.5, "Нефтепродукты": 9.2, "Зерно": 5.1,
           "Металл": 6.8, "Лес": 4.6, "Стройматериалы": 5.9}
clients = ["Агрохолдинг Нива", "МеталлПром", "СтройБаза 77", "НефтеТрейд", "ЛесЭкспорт", "РитейлСеть"]


def make_trips(n, month, start_id):
    rows = []
    for i in range(start_id, start_id + n):
        route, distance = random.choice(list(routes.items()))
        cargo = random.choice(list(tariffs))
        weight = round(random.uniform(3, 22), 1)
        delay = 0 if random.random() < 0.72 else random.choice([1, 2, 3, 4, 6, 8])
        rows.append({
            "trip_id": f"T{i:04d}",
            "date": f"2025-{month:02d}-{random.randint(1, 28):02d}",
            "route": route,
            "carrier": random.choice(carriers),
            "cargo_type": cargo,
            "client": random.choice(clients),
            "weight_t": weight,
            "distance_km": distance,
            "cost_rub": int(round(distance * weight * tariffs[cargo] * random.uniform(0.95, 1.08), -1)),
            "delay_h": delay,
        })
    return rows


def save(path, rows, fields=None):
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields or list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)


april = make_trips(250, 4, 1)
may = make_trips(280, 5, 1001)

# в майской выгрузке один рейс приписан перевозчику, которого нет в справочнике
may[7]["carrier"] = "НовыйПеревозчик"

save("data/trips_april.csv", april)
save("data/trips_may.csv", may)

# Справочник перевозчиков. Внимание: одна строка случайно задвоена —
# это пригодится, когда будем проверять соединение таблиц.
carriers_rows = [
    {"carrier": "ТрансЛогистик", "city": "Москва", "fleet": 48, "rating": 4.6},
    {"carrier": "РусКарго", "city": "Москва", "fleet": 35, "rating": 4.1},
    {"carrier": "СибПуть", "city": "Новосибирск", "fleet": 22, "rating": 3.8},
    {"carrier": "ВолгаТранс", "city": "Казань", "fleet": 29, "rating": 4.4},
    {"carrier": "АвтоДорСервис", "city": "Воронеж", "fleet": 18, "rating": 3.5},
    {"carrier": "ВолгаТранс", "city": "Казань", "fleet": 29, "rating": 4.4},
]
save("data/carriers.csv", carriers_rows)

# Справочник клиентов: есть клиент, который в рейсах не встречается
clients_rows = [
    {"client": "Агрохолдинг Нива", "segment": "Агро", "manager": "Петрова", "since": 2019},
    {"client": "МеталлПром", "segment": "Промышленность", "manager": "Иванов", "since": 2017},
    {"client": "СтройБаза 77", "segment": "Строительство", "manager": "Иванов", "since": 2021},
    {"client": "НефтеТрейд", "segment": "Энергетика", "manager": "Соколов", "since": 2018},
    {"client": "ЛесЭкспорт", "segment": "Лес", "manager": "Петрова", "since": 2020},
    {"client": "РитейлСеть", "segment": "Ритейл", "manager": "Соколов", "since": 2022},
    {"client": "АгроЭкспорт Юг", "segment": "Агро", "manager": "Петрова", "since": 2024},
]
save("data/clients.csv", clients_rows)

# План по маршрутам на май: ставился до начала месяца, поэтому близок к факту, но не равен ему
fact_by_route = {}
for row in may:
    key = row["route"]
    agg = fact_by_route.setdefault(key, {"trips": 0, "revenue": 0})
    agg["trips"] += 1
    agg["revenue"] += row["cost_rub"]

plan_rows = []
for route in routes:
    agg = fact_by_route.get(route, {"trips": 30, "revenue": 1500000})
    plan_rows.append({
        "route": route,
        "month": "2025-05",
        "plan_trips": max(1, round(agg["trips"] * random.uniform(0.85, 1.2))),
        "plan_revenue": int(round(agg["revenue"] * random.uniform(0.85, 1.2), -3)),
    })
save("data/plan.csv", plan_rows)

print("Готово. Создано:")
print("  data/trips_april.csv — рейсы за апрель:", len(april), "строк")
print("  data/trips_may.csv   — рейсы за май:   ", len(may), "строк")
print("  data/carriers.csv    — справочник перевозчиков:", len(carriers_rows), "строк")
print("  data/clients.csv     — справочник клиентов:    ", len(clients_rows), "строк")
print("  data/plan.csv        — план по маршрутам:      ", len(plan_rows), "строк")

---
# Часть 1. `concat` — склейка таблиц

`concat` ставит таблицы **друг под друга**. Применяется, когда структура одинаковая,
а данные относятся к разным периодам или источникам: месяц к месяцу, филиал к филиалу.

Два обязательных параметра на практике:

| Параметр | Зачем |
|---|---|
| `ignore_index=True` | сквозная нумерация: иначе индексы повторятся (0,1,2,... 0,1,2,...) |
| `keys=[...]` | пометить, откуда пришла строка |

И обязательная проверка после склейки: **сумма строк исходных таблиц должна совпасть
с числом строк результата**.

In [ ]:
import pandas as pd

april = pd.read_csv("data/trips_april.csv", parse_dates=["date"])
may = pd.read_csv("data/trips_may.csv", parse_dates=["date"])

print("апрель:", april.shape, "| май:", may.shape)
print("столбцы совпадают:", list(april.columns) == list(may.columns))
print()

trips = pd.concat([april, may], ignore_index=True)

print("после склейки:", trips.shape)
print("проверка:", len(april), "+", len(may), "=", len(trips),
      "->", "OK" if len(april) + len(may) == len(trips) else "ОШИБКА")
print("дубликатов trip_id:", trips["trip_id"].duplicated().sum())

In [ ]:
# Удобный признак месяца — он понадобится во всех группировках
trips["month"] = trips["date"].dt.to_period("M").astype(str)
print(trips["month"].value_counts().sort_index())
print()

# Если нужно помнить источник строки — параметр keys
marked = pd.concat([april, may], keys=["апрель", "май"], names=["источник", "строка"])
print(marked.index[:3].tolist())

> **Что будет, если структура таблиц не совпадает.** `concat` не упадёт: он объединит
> все столбцы, а отсутствующие заполнит `NaN`. Выглядит как «всё получилось», а на деле
> у половины строк пустой столбец. Поэтому сравнение `list(a.columns) == list(b.columns)`
> перед склейкой — не формальность.

### Задание 1

1. Склейте апрель и май в таблицу `trips` со сквозной нумерацией. Проверьте число строк.
2. Добавьте столбец `month`.
3. Проверьте, что номера рейсов не повторяются. Почему это важно проверить именно
   **после** склейки?
4. Сымитируйте проблему: удалите из копии майской таблицы столбец `delay_h`
   (`may.drop(columns=["delay_h"])`), склейте с апрелем и посмотрите, сколько получилось
   пропусков в `delay_h`. Какой вывод?

In [ ]:
# 1)
trips = ...
print("строк:", ...)

# 2)
trips["month"] = ...

# 3)
print("дубликатов trip_id:", ...)

# 4)
broken = ...
print("пропусков в delay_h:", ...)

---
# Часть 2. Сортировка, фильтрация, преобразование

Базовые операции, которые понадобятся в каждой задаче.

| Задача | Код |
|---|---|
| сортировка по одному столбцу | `df.sort_values("cost_rub", ascending=False)` |
| сортировка по нескольким | `df.sort_values(["route", "cost_rub"], ascending=[True, False])` |
| топ-N без сортировки всей таблицы | `df.nlargest(10, "cost_rub")` |
| фильтр | `df[(df["a"] > 1) & (df["b"] < 2)]` |
| фильтр строкой | `df.query("distance_km > 500 and delay_h > 0")` |
| новый столбец в цепочке | `df.assign(cost_per_km=lambda d: d["cost_rub"] / d["distance_km"])` |
| переименовать / удалить | `df.rename(columns={...})`, `df.drop(columns=[...])` |

`query` и `assign` удобны тем, что их можно выстроить в одну цепочку — код читается
сверху вниз как список действий.

In [ ]:
# Топ-3 самых дорогих рейсов
print(trips.nlargest(3, "cost_rub")[["trip_id", "route", "cargo_type", "cost_rub"]])
print()

# Сортировка по двум столбцам: маршрут по алфавиту, внутри — по убыванию стоимости
print(trips.sort_values(["route", "cost_rub"], ascending=[True, False])
           .head(4)[["route", "trip_id", "cost_rub"]])

In [ ]:
# Один и тот же фильтр двумя способами
mask_way = trips[(trips["distance_km"] > 500) & (trips["delay_h"] > 0)]
query_way = trips.query("distance_km > 500 and delay_h > 0")

print("маской:", len(mask_way), "| query:", len(query_way))
print()

# Цепочка: фильтр -> новый признак -> сортировка -> нужные столбцы
result = (trips
          .query("month == '2025-05'")
          .assign(cost_per_km=lambda d: (d["cost_rub"] / d["distance_km"]).round(2))
          .sort_values("cost_per_km", ascending=False)
          [["trip_id", "route", "cargo_type", "cost_per_km"]]
          .head(5))
print(result)

### Задание 2

1. Выведите 10 самых дорогих рейсов за май — только `trip_id`, `route`, `cargo_type`, `cost_rub`.
2. Отсортируйте рейсы по перевозчику (по алфавиту), внутри каждого — по убыванию стоимости.
   Выведите первые 10 строк.
3. Сколько рейсов с грузом `Нефтепродукты` на маршрутах длиннее 600 км? Сделайте двумя
   способами — маской и через `query` — и сверьте числа.
4. Одной цепочкой: возьмите апрельские рейсы, добавьте признак `cost_per_ton`
   (стоимость за тонну), оставьте рейсы дороже 3000 руб. за тонну, отсортируйте по убыванию.
   Сколько таких рейсов?

In [ ]:
# 1)
...

# 2)
...

# 3)
print("маской:", ..., "| query:", ...)

# 4)
...

---
# Часть 3. `groupby` и агрегирующие функции

`groupby` — главный инструмент аналитика. Он делает то, что на практике 2 мы писали
вручную циклом со словарём-накопителем: разбивает таблицу на группы и считает по каждой.

**Схема работы:** разбить по ключу → применить функцию → собрать результат.

| Запись | Что делает |
|---|---|
| `df.groupby("route")["cost_rub"].sum()` | одна функция к одному столбцу |
| `df.groupby("route")["cost_rub"].agg(["sum", "mean", "count"])` | несколько функций |
| `df.groupby(["month", "carrier"])[...]` | группировка по двум ключам |
| `df.groupby("route").agg(revenue=("cost_rub", "sum"))` | **именованная агрегация** — лучший способ |
| `df.groupby("route").size()` | число строк в группе |
| `.reset_index()` | вернуть ключ группировки из индекса в столбец |

Именованная агрегация читается лучше всего: сразу видно, как будет называться столбец
и из чего он считается.

In [ ]:
# Простейший случай
print(trips.groupby("route")["cost_rub"].sum().sort_values(ascending=False))
print()

# Несколько показателей сразу — именованная агрегация
by_route = (trips.groupby("route")
            .agg(trips_count=("trip_id", "count"),
                 revenue=("cost_rub", "sum"),
                 avg_weight=("weight_t", "mean"),
                 max_cost=("cost_rub", "max"))
            .round(2)
            .sort_values("revenue", ascending=False))
print(by_route)

In [ ]:
# Своя функция внутри агрегации: доля рейсов с задержкой
by_carrier = (trips.groupby("carrier")
              .agg(trips_count=("trip_id", "count"),
                   revenue=("cost_rub", "sum"),
                   delayed_share=("delay_h", lambda s: (s > 0).mean()))
              .round(3)
              .sort_values("revenue", ascending=False))
print(by_carrier)
print()

# Группировка по двум ключам даёт двухуровневый индекс
by_two = trips.groupby(["month", "carrier"])["cost_rub"].sum()
print(by_two.head(6))
print()
print("тот же результат в виде обычной таблицы:")
print(by_two.reset_index().head(3))

## `size` против `count`

Разница незаметна, пока в данных нет пропусков, и становится важной, когда они есть:

- `size()` — сколько **строк** в группе;
- `count()` — сколько **непустых значений** в конкретном столбце.

## `transform` — вернуть результат в исходную таблицу

`agg` сворачивает группу в одну строку. `transform` считает по группе, но возвращает
значение **для каждой строки** — удобно, когда нужна доля или сравнение со средним по группе.

In [ ]:
print("size (строк в группе):")
print(trips.groupby("carrier").size())
print()

# transform: какую долю рейс занимает в выручке своего маршрута
trips["route_revenue"] = trips.groupby("route")["cost_rub"].transform("sum")
trips["share_in_route"] = (trips["cost_rub"] / trips["route_revenue"] * 100).round(2)

print(trips[["trip_id", "route", "cost_rub", "route_revenue", "share_in_route"]].head(5))
print()
print("сумма долей по одному маршруту:",
      trips[trips["route"] == "Москва-Казань"]["share_in_route"].sum().round(1), "%")

### Задание 3

1. Посчитайте по каждому **перевозчику**: число рейсов, суммарную выручку, средний вес,
   среднюю задержку и долю рейсов с задержкой. Используйте именованную агрегацию.
   Отсортируйте по выручке.
2. Посчитайте выручку по парам «месяц — тип груза». Приведите результат к обычной таблице
   через `reset_index()`.
3. У какого перевозчика самая высокая **средняя стоимость рейса**? А у какого — самая
   высокая **суммарная**? Это один и тот же перевозчик? Объясните результат.
4. Через `transform` добавьте столбец `vs_route_avg` — во сколько раз стоимость рейса
   отличается от средней по его маршруту. Найдите рейс с максимальным значением.
5. Сравните `groupby("client").size()` и `groupby("client")["delay_h"].count()`.
   Совпали? Почему?

In [ ]:
# 1)
by_carrier = ...
print(by_carrier)

# 2)
by_month_cargo = ...

# 3)
...

# 4)
trips["vs_route_avg"] = ...

# 5)
...

---
# Часть 4. `merge` — соединение по ключу

`merge` присоединяет к таблице столбцы **из другой таблицы** по общему ключу.
Это аналог `VLOOKUP` в Excel и `JOIN` в SQL.

Четыре типа соединения — параметр `how`:

| `how` | Что остаётся в результате |
|---|---|
| `"left"` | все строки левой таблицы; не нашлось справа — `NaN` |
| `"inner"` | только те, что нашлись в обеих |
| `"right"` | все строки правой |
| `"outer"` | все строки из обеих |

**Для аналитики по умолчанию — `how="left"`.** Левая таблица это ваши факты (рейсы),
и терять их нельзя: если перевозчика нет в справочнике, рейс всё равно был.

Два параметра, которые спасают от тихих ошибок:

- `indicator=True` — добавляет столбец с пометкой, откуда взялась строка;
- `validate="m:1"` — проверка «много к одному»: падает с ошибкой, если ключ в правой
  таблице не уникален.

In [ ]:
carriers = pd.read_csv("data/carriers.csv")
clients = pd.read_csv("data/clients.csv")

print("справочник перевозчиков:")
print(carriers)
print()
print("строк:", len(carriers), "| уникальных перевозчиков:", carriers["carrier"].nunique())

Посмотрите на последнюю строку вывода внимательно: строк шесть, а перевозчиков пять.
Запомните это и запустите следующую ячейку.

In [ ]:
# Присоединяем справочник "как есть"
merged = trips.merge(carriers, on="carrier", how="left")

print("строк в рейсах было: ", len(trips))
print("строк после merge:   ", len(merged))
print("прибавилось:         ", len(merged) - len(trips))
print()
print(f"выручка до merge:    {trips['cost_rub'].sum():>12,}".replace(",", " "))
print(f"выручка после merge: {merged['cost_rub'].sum():>12,}".replace(",", " "))
print(f"завышение:           {merged['cost_rub'].sum() - trips['cost_rub'].sum():>12,}".replace(",", " "))

Выручка выросла на 3.7 миллиона, хотя мы всего лишь присоединили справочник с рейтингами.
**Ни одной ошибки не было.**

Причина: в справочнике перевозчик `ВолгаТранс` записан дважды. Для каждого рейса этого
перевозчика `merge` нашёл **два** подходящих значения справа — и создал две строки вместо
одной. Все 92 рейса `ВолгаТранс` задвоились, и их стоимость попала в сумму дважды.

Это самая частая и самая дорогая ошибка при соединении таблиц. Защита — две строки:

In [ ]:
# 1. Проверка "много к одному": падает сразу, если ключ справа не уникален
try:
    trips.merge(carriers, on="carrier", how="left", validate="m:1")
except Exception as e:
    print(type(e).__name__, "->", e)
print()

# 2. Лечение: убрать дубли в справочнике
carriers_ok = carriers.drop_duplicates(subset="carrier")
merged = trips.merge(carriers_ok, on="carrier", how="left", validate="m:1")

print("строк после merge:", len(merged), "| было:", len(trips),
      "->", "OK" if len(merged) == len(trips) else "ОШИБКА")
print(f"выручка: {merged['cost_rub'].sum():,}".replace(",", " "), "- совпадает с исходной")

In [ ]:
# Кто не нашёлся в справочнике
print("рейсов без данных о перевозчике:", merged["rating"].isna().sum())
print("это перевозчик:", merged.loc[merged["rating"].isna(), "carrier"].unique().tolist())
print()

# indicator показывает, с какой стороны пришла каждая строка
check = trips.merge(clients, on="client", how="outer", indicator=True)
print(check["_merge"].value_counts())
print()
print("есть в справочнике, но рейсов не было:",
      check.loc[check["_merge"] == "right_only", "client"].unique().tolist())

Два вида несовпадений, и оба нужно замечать:

- **`left_only`** — факт есть, справочника нет. У нас это рейс перевозчика
  `НовыйПеревозчик`: рейтинга мы не знаем, но рейс состоялся и выручку принёс.
  Выбрасывать нельзя.
- **`right_only`** — справочник есть, фактов нет. Клиент `АгроЭкспорт Юг` в договоре
  числится, но ни одного рейса за два месяца. Для отчёта по выручке он не нужен,
  а вот менеджеру по работе с клиентами это интересно.

### Задание 4

1. Присоедините к `trips` справочник перевозчиков — правильно, с проверкой числа строк.
2. Присоедините справочник клиентов (`how="left"`). Проверьте число строк снова.
3. Сколько рейсов осталось без `segment`? Почему?
4. Сделайте то же соединение с `how="inner"` и сравните число строк с `how="left"`.
   Какие рейсы потерялись и почему это плохо для отчёта о выручке?
5. Посчитайте выручку по сегментам клиентов. Какой сегмент крупнейший?

In [ ]:
# 1)
full = ...
print("строк:", ..., "| было:", len(trips))

# 2)
full = ...
print("строк:", ...)

# 3)
print("рейсов без сегмента:", ...)

# 4)
inner = ...
print("left:", ..., "| inner:", ...)

# 5)
...

---
# Часть 5. `pivot_table` и сводные представления

Сводная таблица — это группировка по **двум** ключам, разложенная в прямоугольник:
один ключ в строки, другой в столбцы. Такой вид удобно читать глазами и показывать
на совещании.

```python
pd.pivot_table(df,
               index="route",        # что в строках
               columns="cargo_type", # что в столбцах
               values="cost_rub",    # что считаем
               aggfunc="sum",        # как считаем
               fill_value=0,         # чем заполнить пустые ячейки
               margins=True)         # добавить строку и столбец "Итого"
```

| Инструмент | Когда |
|---|---|
| `pivot_table` | нужны числа по двум измерениям с агрегацией |
| `crosstab` | нужно просто посчитать количество пересечений |
| `pivot` | данные уже агрегированы, нужно только «развернуть» |

In [ ]:
full = trips.merge(carriers_ok, on="carrier", how="left", validate="m:1")
full = full.merge(clients, on="client", how="left")

pivot = pd.pivot_table(full,
                       index="route",
                       columns="cargo_type",
                       values="cost_rub",
                       aggfunc="sum",
                       fill_value=0,
                       margins=True,
                       margins_name="Итого")

print("Выручка по маршрутам и типам груза:")
print(pivot.iloc[:4, :4])
print()
print("общая выручка (угловая ячейка):", f"{pivot.loc['Итого', 'Итого']:,}".replace(",", " "))

In [ ]:
# Выручка по сегментам и месяцам
print(pd.pivot_table(full, index="segment", columns="month",
                     values="cost_rub", aggfunc="sum", fill_value=0))
print()

# Две функции сразу
print(pd.pivot_table(full, index="carrier", values="cost_rub",
                     aggfunc=["count", "sum", "mean"]).round(0))

In [ ]:
# crosstab: просто количество рейсов на пересечении
print("Число рейсов: перевозчик x месяц")
print(pd.crosstab(full["carrier"], full["month"], margins=True, margins_name="Итого"))
print()

# normalize превращает количество в доли
print("Доли по строкам:")
print(pd.crosstab(full["carrier"], full["month"], normalize="index").round(3))

### Задание 5

1. Постройте сводную: выручка по **перевозчикам** (строки) и **месяцам** (столбцы),
   с итогами.
2. Постройте сводную: **средняя задержка** по маршрутам и типам груза. На пересечении
   каких маршрута и груза задержки самые большие?
3. Через `crosstab` посчитайте число рейсов по парам «сегмент клиента — тип груза».
4. Постройте сводную по менеджерам (`manager`): число рейсов и выручка. Кто ведёт
   больше всего выручки?
5. Постройте сводную **средняя задержка по перевозчикам и типам груза**. В ней есть
   и пустые ячейки, и ячейки со значением `0.00`. Чем они отличаются по смыслу?
   Как проверить догадку? И почему здесь **нельзя** писать `fill_value=0`?

In [ ]:
# 1)
...

# 2)
...

# 3)
...

# 4)
...

# 5)
...

---
# Часть 6. Мини-проект: план/факт по маршрутам

Руководитель просит таблицу выполнения плана за май: по каждому маршруту — план, факт
и процент выполнения.

Это полный цикл сегодняшней пары: отфильтровать период → сгруппировать факт →
присоединить план → посчитать показатели → отсортировать и показать.

### Задание 6

1. Прочитайте `data/plan.csv`.
2. Посчитайте факт за май по маршрутам: число рейсов и выручку.
3. Присоедините план к факту. Какой `how` выбрать, чтобы в отчёте оказались **все**
   маршруты из плана, даже если по ним не было ни одного рейса? Проверьте число строк.
4. Посчитайте `trips_done_pct` и `revenue_done_pct` — выполнение плана в процентах.
5. Отсортируйте по выполнению плана по выручке, выведите аккуратную таблицу.
6. Посчитайте итог: суммарный план, суммарный факт, общее выполнение в процентах.

In [ ]:
plan = pd.read_csv("data/plan.csv")
print(plan)

# 2)
fact = ...

# 3)
pf = ...
print("строк в плане:", len(plan), "| строк после соединения:", ...)

# 4)
pf["trips_done_pct"] = ...
pf["revenue_done_pct"] = ...

# 5)
...

# 6)
...

### Задание 7. Выводы для руководителя

Ответьте по своей таблице:

1. Сколько маршрутов выполнили план по выручке? Сколько не выполнили?
2. Найдите маршрут, где **рейсов сделано больше плана, а выручка ниже плана**.
   Как такое возможно? Что это говорит о структуре перевозок на этом маршруте?
3. Общее выполнение плана по компании — выше или ниже 100%?
4. Напишите три предложения для руководителя: что с планом, где проблема, что проверить.

> **Выводы:**
>
> 1.
> 2.
> 3.

---
## Отправляем работу

```
# терминал
git add .
git commit -m "Практика 9: объединение и агрегирование, план/факт по маршрутам"
git push
```

## Чек-лист сдачи

- [ ] Задание 1: таблицы склеены, число строк проверено, показан эффект несовпадения столбцов
- [ ] Задание 2: сортировки, фильтры двумя способами, цепочка с `assign`
- [ ] Задание 3: именованная агрегация, группировка по двум ключам, `transform`
- [ ] Задание 4: `merge` с проверкой числа строк, разобраны `left_only` и `right_only`
- [ ] Задание 5: пять сводных таблиц, объяснены пустые ячейки
- [ ] Задание 6: таблица план/факт с процентами выполнения
- [ ] Задание 7: письменные выводы
- [ ] Всё запушено в репозиторий

## Что унести с пары

`groupby` и `pivot_table` — это удобство: то же самое можно было посчитать циклом, просто
дольше. А вот `merge` — это риск. Он единственный из сегодняшних инструментов умеет
**менять число строк**, причём молча: никакой ошибки, просто выручка выросла на 17%.

Поэтому запомните две строки, которые должны идти за каждым соединением:

```python
assert len(result) == len(left_table)        # число строк не изменилось
print(result["key_column"].isna().sum())     # сколько не нашло пары
```

Если число строк изменилось, а вы этого не ждали — ключ в правой таблице не уникален.
Сначала разберитесь с этим, потом считайте суммы.